Raymond Chi & Arav Sharma

In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

data = pd.read_csv('ml-100k/u.data', sep='\t', header=None, names=['user_id', 'item_id', 'rating', 'timestamp'])
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)
print(train_data.head(), train_data.shape, test_data.shape)


       user_id  item_id  rating  timestamp
75220      807     1411       1  893082619
48955      474      659       5  887925187
44966      463      268       4  877384940
13568      139      286       4  879537844
92727      621      751       4  883799651 (80000, 4) (20000, 4)


In [51]:
n_users = data['user_id'].unique().shape[0]
n_movies = data['item_id'].unique().shape[0]
print(n_users, n_movies)

R_train = np.zeros((n_users, n_movies))
R_test = np.zeros((n_users, n_movies))

print(R_train.shape, R_test.shape)

for row in train_data.itertuples():
    R_train[row.user_id - 1, row.item_id - 1] = row.rating
for row in test_data.itertuples():
    R_test[row.user_id - 1, row.item_id - 1] = row.rating

print(R_train.shape, R_test.shape)

943 1682
(943, 1682) (943, 1682)
(943, 1682) (943, 1682)


In [53]:
def nmf_als(R, k, lambda_reg, num_iters, early_stopping_rounds=5):
    m, n = R.shape
    p = np.random.rand(m, k) * 0.1
    q = np.random.rand(n, k) * 0.1
    
    prev_loss = float('inf')
    mask = R > 0
    best_rmse = float('inf')
    patience_counter = 0

### Alternating lest squares
    
    for i in range(num_iters):
        # Update p
        for u in range(m): # loop over each users
            Rating_u = R[u, :] #  get the ratings of user u
            q_nonzero = q[Rating_u > 0] # get the movies that user u has rated
            Rating_u_nonzero = Rating_u[Rating_u > 0]
            if len(Rating_u_nonzero) > 0:
                A = q_nonzero.T @ q_nonzero + lambda_reg * np.eye(k) # https://medium.com/@rinabuoy13/explicit-recommender-system-matrix-factorization-in-pytorch-f3779bb55d74
                b = q_nonzero.T @ Rating_u_nonzero
                p[u] = np.maximum(np.linalg.solve(A, b), 0)
        
        # Update q
        for j in range(n): # loop over each movies
            Rating_i = R[:, j] # get the ratings of movie j
            P_nonzero = p[Rating_i > 0] # get the users that have rated movie
            Rating_i_nonzero = Rating_i[Rating_i > 0]
            if len(Rating_i_nonzero) > 0:
                A = P_nonzero.T @ P_nonzero + lambda_reg * np.eye(k) # https://medium.com/@rinabuoy13/explicit-recommender-system-matrix-factorization-in-pytorch-f3779bb55d74  
                b = P_nonzero.T @ Rating_i_nonzero
                q[j] = np.maximum(np.linalg.solve(A, b), 0)
        
        # Compute metrics
        pred = p @ q.T
        rmse = np.sqrt(np.sum(mask * (R - pred) ** 2) / np.sum(mask))
        
        if i % 20 == 0:
            print(f"Iteration {i}, RMSE: {rmse:.4f}")
            
        # Early stopping
        if rmse < best_rmse:
            best_rmse = rmse
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_rounds:
                print(f"Early stopping at iteration {i}")
                break
    
    return p, q

In [ ]:
def cross_validate_nmf(R_train, k_values, lambda_values, num_iters=100, n_splits=3):

    from sklearn.model_selection import KFold
    from sklearn.metrics import mean_squared_error

    n_users = R_train.shape[0]
    user_indices = np.arange(n_users)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    best_rmse = float('inf')
    best_k = None
    best_lambda = None
    
    for k in k_values:
        for lambda_reg in lambda_values:
            fold_rmses = []
            
            for train_idx, val_idx in kf.split(user_indices):
                R_train_fold = R_train.copy()
                R_val_fold = np.zeros_like(R_train)
                
                # Split validation data
                for user_idx in val_idx:
                    rated_items = np.where(R_train[user_idx] > 0)[0]
                    if len(rated_items) > 0:
                        n_val = max(1, int(0.2 * len(rated_items)))
                        val_items = np.random.choice(rated_items, n_val, replace=False)
                        R_val_fold[user_idx, val_items] = R_train_fold[user_idx, val_items]
                        R_train_fold[user_idx, val_items] = 0
                
                # Train and evaluate
                P, Q = nmf_als(R_train_fold, k, lambda_reg, num_iters)
                R_pred = P @ Q.T
                val_mask = R_val_fold > 0
                rmse = np.sqrt(mean_squared_error(R_val_fold[val_mask], R_pred[val_mask]))
                fold_rmses.append(rmse)
            
            avg_rmse = np.mean(fold_rmses)
            if avg_rmse < best_rmse:
                best_rmse = avg_rmse
                best_k = k
                best_lambda = lambda_reg
    
    return best_k, best_lambda, best_rmse

the rmse is generally lower than 2.5 around 0.8 - 1.8.

In [56]:
from  sklearn.metrics import mean_squared_error

k_values = [5, 10, 15, 20]
lambda_values = [0.01, 0.1, 1.0]

best_k, best_lambda, best_cv_rmse = cross_validate_nmf(R_train, k_values, lambda_values)
print(f"\nBest parameters: k={best_k}, lambda={best_lambda}, CV RMSE={best_cv_rmse:.4f}")

# Train final model
P, Q = nmf_als(R_train, best_k, best_lambda, num_iters=100)

# Evaluate on test set
R_pred = P @ Q.T
test_mask = R_test > 0
test_rmse = np.sqrt(mean_squared_error(R_test[test_mask], R_pred[test_mask]))
print(f"Final Test RMSE: {test_rmse:.4f}")

Iteration 0, RMSE: 1.6047
Iteration 20, RMSE: 0.8259
Early stopping at iteration 23
Iteration 0, RMSE: 1.7889
Iteration 20, RMSE: 0.8939
Early stopping at iteration 20
Iteration 0, RMSE: 2.0384
Iteration 20, RMSE: 0.8332
Early stopping at iteration 20
New best model - k: 5, lambda: 0.01, RMSE: 1.9011
Iteration 0, RMSE: 1.9603
Iteration 20, RMSE: 0.8271
Iteration 40, RMSE: 0.8150
Early stopping at iteration 42
Iteration 0, RMSE: 2.0089
Iteration 20, RMSE: 0.8287
Iteration 40, RMSE: 0.8106
Iteration 60, RMSE: 0.8030
Early stopping at iteration 66
Iteration 0, RMSE: 2.0421
Iteration 20, RMSE: 0.8298
Iteration 40, RMSE: 0.8124
Iteration 60, RMSE: 0.8025
Early stopping at iteration 72
New best model - k: 5, lambda: 0.1, RMSE: 1.0747
Iteration 0, RMSE: 5.9306
Iteration 20, RMSE: 0.7950
Iteration 40, RMSE: 0.7835
Iteration 60, RMSE: 0.7814
Iteration 80, RMSE: 0.7804
Iteration 0, RMSE: 5.2295
Iteration 20, RMSE: 0.7891
Iteration 40, RMSE: 0.7812
Iteration 60, RMSE: 0.7798
Iteration 80, RMSE: 0